# Notebook 09a — Análisis descriptivo económico de la cartera

## Objetivo

Realizar el análisis descriptivo de la cartera de clientes de Selmark con foco en la composición general, la identificación de cuentas técnicas y la caracterización económica de los canales B2B y retail. Este notebook constituye la primera parte del análisis del Capítulo 4 de la memoria del TFG y se complementa con el notebook 09b, dedicado a los análisis temporal y sociodemográfico.

## Estructura

1. Configuración y carga de gold.cliente_360
2. Visión general de la cartera (3.492 clientes)
3. Identificación de cuentas técnicas
4. Análisis económico B2B
5. Análisis económico Retail
6. Cruce B2B × Retail
7. Síntesis económica

## Decisiones metodológicas

Los análisis se presentan en dos vistas cuando procede: con todas las cuentas y excluyendo las cuentas técnicas REGO, HERREROS y EL CORTE INGLES. Esta doble lectura permite caracterizar el negocio real sin la distorsión de las cuentas operativas, manteniendo al mismo tiempo la trazabilidad de su impacto. Los gráficos se generan con plotly para permitir la exploración interactiva y se exportan adicionalmente como PNG estáticas en `output/figuras/09a/` para su inclusión en la memoria.

## 1. Configuración y carga de la tabla analítica

In [20]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Rutas del proyecto
RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"
RUTA_FIGURAS = RUTA_PROYECTO / "output" / "figuras" / "09a"
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Paleta corporativa (sector moda íntima femenina)
COLOR_NACIONAL       = "#C8447F"  # rosa empolvado
COLOR_INTERNACIONAL  = "#3E5C76"  # azul marino suave
COLOR_DESTACAR       = "#D4A017"  # dorado
COLOR_POSITIVO       = "#5C9E76"  # verde sage
COLOR_NEGATIVO       = "#B85450"  # rojo apagado
COLOR_NEUTRO         = "#7D7D7D"  # gris medio
COLOR_FONDO          = "#FAFAFA"

PALETA_CATEGORICA = [
    "#C8447F", "#3E5C76", "#D4A017", "#5C9E76",
    "#B85450", "#7D7D7D", "#9B7BA6", "#E08E45"
]

# Plantilla plotly personalizada
TEMPLATE = go.layout.Template(
    layout=dict(
        font=dict(family="Arial, Helvetica, sans-serif", size=12, color="#1A1A1A"),
        title=dict(font=dict(size=16, color="#1A1A1A"), x=0.02, xanchor="left"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        colorway=PALETA_CATEGORICA,
        xaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False,
                   linecolor="#333333", linewidth=0.8, ticks="outside"),
        yaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False,
                   linecolor="#333333", linewidth=0.8, ticks="outside"),
        margin=dict(l=60, r=40, t=70, b=60),
        legend=dict(bgcolor="rgba(255,255,255,0.95)", bordercolor="#CCCCCC",
                    borderwidth=0.5),
    )
)
pio.templates["selmark"] = TEMPLATE
pio.templates.default = "selmark"

# Conexión a DuckDB en modo lectura
con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

# Función auxiliar: guardar PNG y mostrar interactivo
def guardar_y_mostrar(fig, nombre, w=900, h=500):
    """Guarda la figura como PNG en RUTA_FIGURAS y la muestra en el notebook."""
    ruta_png = RUTA_FIGURAS / f"{nombre}.png"
    fig.write_image(str(ruta_png), width=w, height=h, scale=2)
    fig.show()
    print(f"   Figura guardada en: {ruta_png.name}")

# Función auxiliar: formato europeo para números
def fmt_es(n, decimales=0):
    """Formatea un número con separador de miles '.' y decimal ','."""
    if pd.isna(n):
        return "—"
    if decimales == 0:
        return f"{int(n):,}".replace(",", ".")
    return f"{n:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Conexión establecida con: {RUTA_DUCKDB.name}")
print(f"Figuras se guardarán en: {RUTA_FIGURAS}")
print(f"Plantilla plotly 'selmark' activada por defecto.")

Conexión establecida con: selmark.duckdb
Figuras se guardarán en: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\output\figuras\09a
Plantilla plotly 'selmark' activada por defecto.


In [21]:
# Carga de la tabla analítica completa
df = con.execute("SELECT * FROM gold.cliente_360").fetchdf()

print(f"Tabla cargada: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas disponibles (primeras 20):")
print(list(df.columns[:20]))
print(f"\n... y {df.shape[1] - 20} columnas más.")

Tabla cargada: 3,492 filas × 56 columnas

Columnas disponibles (primeras 20):
['id_cliente', 'nombre_cliente', 'nombre_comercial_cliente', 'localidad_cliente', 'codigo_postal_norm', 'codigo_provincia_cliente', 'tipo_mercado', 'es_cliente_espanol', 'pais', 'mosaic_grupo', 'mosaic_grupo_peso', 'mosaic_segmento', 'mosaic_segmento_peso', 'renta_media', 'perfil_premium', 'perfil_familiar_joven', 'perfil_turistico', 'perfil_rural', 'perfil_precio_sensible', 'tiene_mosaic']

... y 36 columnas más.


## 2. Visión general de la cartera

Se presenta a continuación la composición global de la cartera: distribución entre los mercados nacional e internacional, geografía, y cobertura por bloque temático de la tabla analítica.

In [22]:
# 2.1 Distribución nacional vs internacional
distrib = df.groupby("tipo_mercado").size().reset_index(name="num_clientes")
distrib["porcentaje"] = (100 * distrib["num_clientes"] / distrib["num_clientes"].sum()).round(2)
distrib = distrib.sort_values("num_clientes", ascending=False)

print("DISTRIBUCIÓN NACIONAL VS INTERNACIONAL")
print("=" * 50)
for _, row in distrib.iterrows():
    print(f"   {row['tipo_mercado']:<15s}  {fmt_es(row['num_clientes']):>7s} clientes  ({row['porcentaje']:>5.2f} %)")

fig = go.Figure(data=[go.Pie(
    labels=distrib["tipo_mercado"],
    values=distrib["num_clientes"],
    marker=dict(colors=[COLOR_NACIONAL, COLOR_INTERNACIONAL]),
    textinfo="label+percent",
    textposition="auto",
    textfont=dict(size=14, color="white"),
    hole=0.5,
)])
fig.update_layout(
    title=dict(text="<b>Composición de la cartera</b><br><sup>Distribución entre los mercados nacional e internacional · 3.492 clientes</sup>"),
    annotations=[dict(text=f"<b>{fmt_es(len(df))}</b><br>clientes", x=0.5, y=0.5,
                       font=dict(size=18), showarrow=False)],
    height=500,
)
guardar_y_mostrar(fig, "01_composicion_cartera")

DISTRIBUCIÓN NACIONAL VS INTERNACIONAL
   NACIONAL           2.105 clientes  (60.28 %)
   INTERNACIONAL      1.387 clientes  (39.72 %)


   Figura guardada en: 01_composicion_cartera.png


In [23]:
# 2.2 Top 15 países de la cartera internacional
top_paises = (df[df["tipo_mercado"] == "INTERNACIONAL"]
              .groupby("pais").size().reset_index(name="num_clientes")
              .sort_values("num_clientes", ascending=False)
              .head(15))

fig = px.bar(
    top_paises.sort_values("num_clientes"),
    x="num_clientes", y="pais",
    orientation="h",
    color_discrete_sequence=[COLOR_INTERNACIONAL],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{y}</b><br>Clientes: %{x:,}<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Top 15 países de la cartera internacional</b><br><sup>Número de clientes por país</sup>"),
    xaxis_title="Número de clientes",
    yaxis_title="",
    height=550, showlegend=False,
)
guardar_y_mostrar(fig, "02_top15_paises_internacional")

   Figura guardada en: 02_top15_paises_internacional.png


### Diagnóstico de clientes internacionales sin país identificado

Durante la investigación documentada en el notebook 03b se confirmó la existencia de un grupo residual de clientes internacionales cuyo código de provincia no pudo asociarse a un país concreto. Se caracterizan a continuación para confirmar su naturaleza residual y su impacto sobre los agregados económicos.

In [24]:
# Diagnóstico de clientes internacionales sin país identificado
print("DIAGNÓSTICO — Clientes internacionales sin país identificado")
print("=" * 75)

mask_sin_pais = (df["tipo_mercado"] == "INTERNACIONAL") & (
    df["pais"].isna() | (df["pais"].astype(str).str.strip().isin(["", "Sin identificar", "—", "None"]))
)

sin_pais = df[mask_sin_pais].copy()
n_sin = len(sin_pais)
total_intl = int((df["tipo_mercado"] == "INTERNACIONAL").sum())
pct_sin = 100 * n_sin / total_intl if total_intl > 0 else 0

print(f"\n   Clientes internacionales totales:     {total_intl:>6,}")
print(f"   Sin país identificado:                 {n_sin:>6,}  ({pct_sin:.2f} %)")
print(f"   Con país identificado:                 {total_intl - n_sin:>6,}  ({100 - pct_sin:.2f} %)")

if n_sin > 0:
    print(f"\n   Distribución por código de provincia (codigo_provincia_cliente):")
    dist_codigos = sin_pais["codigo_provincia_cliente"].value_counts().head(15)
    for codigo, n in dist_codigos.items():
        print(f"      Código {str(codigo):<10s}  {n:>4} clientes")

    print(f"\n   Caracterización económica del grupo sin país identificado:")
    print(f"      Con facturación B2B:        {int(sin_pais['tiene_facturacion_b2b'].sum()):>4}  ({100*sin_pais['tiene_facturacion_b2b'].mean():.1f} %)")
    print(f"      Con actividad retail:       {int(sin_pais['tiene_actividad_minorista'].sum()):>4}  ({100*sin_pais['tiene_actividad_minorista'].mean():.1f} %)")
    print(f"      Facturación B2B total:      {fmt_es(sin_pais['facturacion_b2b'].sum(), 2):>15s} €  ({100*sin_pais['facturacion_b2b'].sum()/df['facturacion_b2b'].sum():.2f} % del total)")
    print(f"      Facturación retail total:   {fmt_es(sin_pais['facturacion_retail_neta_eur'].sum(), 2):>15s} €  ({100*sin_pais['facturacion_retail_neta_eur'].sum()/df['facturacion_retail_neta_eur'].sum():.2f} % del total)")

    sin_pais["facturacion_combinada"] = sin_pais["facturacion_b2b"] + sin_pais["facturacion_retail_neta_eur"]
    top10_sin_pais = sin_pais.nlargest(10, "facturacion_combinada")[
        ["id_cliente", "nombre_cliente", "localidad_cliente", "codigo_provincia_cliente",
         "facturacion_b2b", "facturacion_retail_neta_eur"]
    ].copy()

    if len(top10_sin_pais) > 0:
        print(f"\n   Top 10 del grupo (por facturación combinada):")
        for c in ["facturacion_b2b", "facturacion_retail_neta_eur"]:
            top10_sin_pais[c] = top10_sin_pais[c].apply(lambda v: fmt_es(v, 2) + " €" if pd.notna(v) else "—")
        print(top10_sin_pais.to_string(index=False, max_colwidth=35))
else:
    print("\n   Todos los clientes internacionales tienen país identificado.")

print(f"\n   Conclusión: estos casos residuales (~{pct_sin:.1f} %) son consecuencia de los")
print(f"   códigos de provincia internacionales que no pudieron asociarse a un país")
print(f"   concreto durante la investigación del notebook 03b. Su impacto sobre los")
print(f"   agregados económicos es marginal y se documenta como caso conocido del modelo.")

DIAGNÓSTICO — Clientes internacionales sin país identificado

   Clientes internacionales totales:      1,387
   Sin país identificado:                     35  (2.52 %)
   Con país identificado:                  1,352  (97.48 %)

   Distribución por código de provincia (codigo_provincia_cliente):
      Código 041            6 clientes
      Código ES             4 clientes

   Caracterización económica del grupo sin país identificado:
      Con facturación B2B:          31  (88.6 %)
      Con actividad retail:         31  (88.6 %)
      Facturación B2B total:           430.844,83 €  (0.51 % del total)
      Facturación retail total:        436.754,49 €  (0.51 % del total)

   Top 10 del grupo (por facturación combinada):
id_cliente                      nombre_cliente      localidad_cliente codigo_provincia_cliente facturacion_b2b facturacion_retail_neta_eur
     16349 JAAF-MALHAS LINGERIE UNIPE, ARMA...      CALDAS DAS TAIPAS                      NaN     72.985,63 €                 84.

### Conclusiones del diagnóstico

El grupo residual de 35 clientes sin país identificado representa el 2,52 % de la cartera internacional y aporta exclusivamente el 0,51 % de la facturación tanto B2B como retail. Estos volúmenes confirman su naturaleza marginal sobre los agregados económicos del proyecto. El análisis nominal del grupo revela que la mayoría corresponde a clientes portugueses, belgas y franceses cuyo código de provincia no estaba contemplado en el mapeo construido en el notebook 03b.

La decisión metodológica adoptada es mantener este grupo en la tabla `gold.cliente_360` sin rescate adicional, documentándolo formalmente como residuo conocido del modelo. El rescate de estos 35 casos requeriría una segunda fase de investigación manual cuyo coste no resulta proporcional al impacto sobre los resultados. Esta decisión se documentará explícitamente en el Capítulo 3 de la memoria como ejemplo de los límites operativos de la integración con sistemas heredados.

In [25]:
# 2.3 Top 15 provincias de la cartera nacional
top_provincias = (df[df["tipo_mercado"] == "NACIONAL"]
                  .groupby("codigo_provincia_cliente").size().reset_index(name="num_clientes")
                  .sort_values("num_clientes", ascending=False)
                  .head(15))

# Mapeo manual de los códigos de provincia más habituales (para enriquecer la lectura)
nombres_provincia = {
    "01":"Álava","02":"Albacete","03":"Alicante","04":"Almería","05":"Ávila","06":"Badajoz",
    "07":"Baleares","08":"Barcelona","09":"Burgos","10":"Cáceres","11":"Cádiz","12":"Castellón",
    "13":"Ciudad Real","14":"Córdoba","15":"A Coruña","16":"Cuenca","17":"Girona","18":"Granada",
    "19":"Guadalajara","20":"Guipúzcoa","21":"Huelva","22":"Huesca","23":"Jaén","24":"León",
    "25":"Lleida","26":"La Rioja","27":"Lugo","28":"Madrid","29":"Málaga","30":"Murcia",
    "31":"Navarra","32":"Ourense","33":"Asturias","34":"Palencia","35":"Las Palmas","36":"Pontevedra",
    "37":"Salamanca","38":"Sta Cruz Tenerife","39":"Cantabria","40":"Segovia","41":"Sevilla",
    "42":"Soria","43":"Tarragona","44":"Teruel","45":"Toledo","46":"Valencia","47":"Valladolid",
    "48":"Vizcaya","49":"Zamora","50":"Zaragoza","51":"Ceuta","52":"Melilla",
}
top_provincias["provincia_nombre"] = top_provincias["codigo_provincia_cliente"].map(
    lambda c: nombres_provincia.get(str(c).zfill(2), f"Cód. {c}"))

fig = px.bar(
    top_provincias.sort_values("num_clientes"),
    x="num_clientes", y="provincia_nombre",
    orientation="h",
    color_discrete_sequence=[COLOR_NACIONAL],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{y}</b><br>Clientes: %{x:,}<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Top 15 provincias de la cartera nacional</b><br><sup>Número de clientes por provincia española</sup>"),
    xaxis_title="Número de clientes",
    yaxis_title="",
    height=550, showlegend=False,
)
guardar_y_mostrar(fig, "03_top15_provincias_nacional")

   Figura guardada en: 03_top15_provincias_nacional.png


In [26]:
# 2.4 Cobertura por bloque temático de la tabla
cobertura = pd.DataFrame({
    "bloque": [
        "Total clientes",
        "Con info MOSAIC",
        "Con facturación B2B",
        "Con actividad retail",
        "Con ambas actividades (B2B + Retail)",
    ],
    "num_clientes": [
        len(df),
        int(df["tiene_mosaic"].sum()) if "tiene_mosaic" in df.columns else int(df["grupo_mosaic"].notna().sum()),
        int(df["tiene_facturacion_b2b"].sum()),
        int(df["tiene_actividad_minorista"].sum()),
        int(((df["tiene_facturacion_b2b"]) & (df["tiene_actividad_minorista"])).sum()),
    ]
})
cobertura["pct_cartera"] = (100 * cobertura["num_clientes"] / len(df)).round(2)

print("COBERTURA POR BLOQUE TEMÁTICO")
print("=" * 65)
for _, row in cobertura.iterrows():
    print(f"   {row['bloque']:<45s}  {fmt_es(row['num_clientes']):>7s}  ({row['pct_cartera']:>5.2f} %)")

fig = px.bar(
    cobertura.iloc[::-1],
    x="num_clientes", y="bloque",
    orientation="h",
    color="pct_cartera", color_continuous_scale=[[0, "#EFD9E5"], [1, COLOR_NACIONAL]],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{y}</b><br>Clientes: %{x:,}<br>Cobertura: %{marker.color:.2f} %<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Cobertura por bloque temático de cliente_360</b><br><sup>Número de clientes con información disponible en cada bloque</sup>"),
    xaxis_title="Número de clientes",
    yaxis_title="",
    height=400, coloraxis_showscale=False,
)
guardar_y_mostrar(fig, "04_cobertura_bloques")

COBERTURA POR BLOQUE TEMÁTICO
   Total clientes                                   3.492  (100.00 %)
   Con info MOSAIC                                  2.038  (58.36 %)
   Con facturación B2B                              3.083  (88.29 %)
   Con actividad retail                             3.054  (87.46 %)
   Con ambas actividades (B2B + Retail)             3.050  (87.34 %)


   Figura guardada en: 04_cobertura_bloques.png


## 3. Identificación de cuentas técnicas

Durante la construcción del modelo se detectaron dos cuentas técnicas con volúmenes de pedidos extraordinariamente elevados para su perfil de persona física (REGO, JAVIER y HERREROS CASTRO, SORAYA), y una cuenta corporativa de gran tamaño (EL CORTE INGLES, S.A.). Estas tres cuentas, aunque legítimas, presentan patrones operativos atípicos que distorsionan los agregados si no se aíslan en el análisis. Esta sección las identifica formalmente y se utiliza más adelante para presentar las vistas con y sin estas cuentas.

In [27]:
# Identificación formal de cuentas técnicas
ids_cuentas_tecnicas = ["5728", "30942", "30957"]  # ECI, REGO, HERREROS

cuentas_tecnicas = df[df["id_cliente"].isin(ids_cuentas_tecnicas)].copy()

print("CUENTAS TÉCNICAS IDENTIFICADAS")
print("=" * 80)
cols_mostrar = ["id_cliente", "nombre_cliente", "tipo_mercado", "num_pedidos_b2b",
                "facturacion_b2b", "num_operaciones", "facturacion_retail_neta_eur"]
cuentas_show = cuentas_tecnicas[cols_mostrar].copy()
for c in ["facturacion_b2b", "facturacion_retail_neta_eur"]:
    cuentas_show[c] = cuentas_show[c].apply(lambda v: fmt_es(v, decimales=2) + " €")
for c in ["num_pedidos_b2b", "num_operaciones"]:
    cuentas_show[c] = cuentas_show[c].apply(lambda v: fmt_es(v) if pd.notna(v) else "—")
print(cuentas_show.to_string(index=False))

# Marcar las cuentas técnicas en el DataFrame
df["es_cuenta_tecnica"] = df["id_cliente"].isin(ids_cuentas_tecnicas)
df_sin_tecnicas = df[~df["es_cuenta_tecnica"]].copy()

print(f"\n   Total cartera:              {len(df):,} clientes")
print(f"   Cuentas técnicas:           {len(cuentas_tecnicas):,} clientes")
print(f"   Cartera sin cuentas técnic: {len(df_sin_tecnicas):,} clientes")

CUENTAS TÉCNICAS IDENTIFICADAS
id_cliente          nombre_cliente tipo_mercado num_pedidos_b2b facturacion_b2b num_operaciones facturacion_retail_neta_eur
     30942            REGO, JAVIER     NACIONAL          50.730  2.891.988,09 €          96.812              3.161.869,54 €
     30957 HERREROS CASTRO, SORAYA     NACIONAL          92.460  2.495.670,19 €         105.318              3.718.968,42 €
      5728   EL CORTE INGLES, S.A.     NACIONAL          13.148 10.812.150,42 €         141.098             10.961.144,91 €

   Total cartera:              3,492 clientes
   Cuentas técnicas:           3 clientes
   Cartera sin cuentas técnic: 3,489 clientes


## 4. Análisis económico B2B

Se examina la distribución de la facturación B2B, la concentración mediante la curva de Pareto, y la comparativa entre los mercados nacional e internacional.

In [28]:
# 4.1 Distribución de la facturación B2B (sin cuentas técnicas, escala log)
b2b = df_sin_tecnicas[df_sin_tecnicas["facturacion_b2b"] > 0].copy()
b2b["log_fact"] = np.log10(b2b["facturacion_b2b"])

fig = px.histogram(
    b2b, x="log_fact", nbins=40, color="tipo_mercado",
    color_discrete_map={"NACIONAL": COLOR_NACIONAL, "INTERNACIONAL": COLOR_INTERNACIONAL},
    barmode="overlay", opacity=0.7,
)
fig.update_layout(
    title=dict(text="<b>Distribución de la facturación B2B</b><br><sup>Histograma en escala logarítmica (sin cuentas técnicas) · 3.080 clientes</sup>"),
    xaxis_title="log₁₀(facturación B2B en euros)",
    yaxis_title="Número de clientes",
    legend=dict(title="Mercado"),
    height=500,
)
fig.add_annotation(text="10 €", x=1, y=-3, showarrow=False, font=dict(color="#888"))
fig.add_annotation(text="100 €", x=2, y=-3, showarrow=False, font=dict(color="#888"))
fig.add_annotation(text="1.000 €", x=3, y=-3, showarrow=False, font=dict(color="#888"))
fig.add_annotation(text="10 K €", x=4, y=-3, showarrow=False, font=dict(color="#888"))
fig.add_annotation(text="100 K €", x=5, y=-3, showarrow=False, font=dict(color="#888"))
fig.add_annotation(text="1 M €", x=6, y=-3, showarrow=False, font=dict(color="#888"))
guardar_y_mostrar(fig, "05_distribucion_facturacion_b2b")

   Figura guardada en: 05_distribucion_facturacion_b2b.png


In [29]:
# 4.2 Curva de Pareto B2B (sin cuentas técnicas)
b2b_sorted = (df_sin_tecnicas[df_sin_tecnicas["facturacion_b2b"] > 0]
              .sort_values("facturacion_b2b", ascending=False).reset_index(drop=True))
b2b_sorted["rank"] = range(1, len(b2b_sorted) + 1)
b2b_sorted["pct_clientes"] = 100 * b2b_sorted["rank"] / len(b2b_sorted)
b2b_sorted["pct_acumulado"] = 100 * b2b_sorted["facturacion_b2b"].cumsum() / b2b_sorted["facturacion_b2b"].sum()

# Hitos clave
hitos = [10, 20, 50, 80]
texto_hitos = []
for h in hitos:
    pct_clientes = h
    n_clientes_h = int(np.ceil(len(b2b_sorted) * h / 100))
    pct_fact = b2b_sorted.iloc[n_clientes_h - 1]["pct_acumulado"]
    texto_hitos.append(f"   Top {h:>3d} % de clientes ({n_clientes_h:>5,}) acumula el {pct_fact:>5.2f} % de la facturación")

print("CURVA DE PARETO B2B (sin cuentas técnicas)")
print("=" * 70)
print("\n".join(texto_hitos))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=b2b_sorted["pct_clientes"], y=b2b_sorted["pct_acumulado"],
    mode="lines", fill="tozeroy",
    line=dict(color=COLOR_NACIONAL, width=3),
    fillcolor="rgba(200, 68, 127, 0.15)",
    name="Facturación acumulada",
    hovertemplate="Top %{x:.1f} %% clientes<br>Acumulan: %{y:.2f} %% facturación<extra></extra>"
))
# Línea diagonal de referencia (equidistribución)
fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100], mode="lines",
    line=dict(color=COLOR_NEUTRO, width=1, dash="dot"),
    name="Equidistribución (referencia)",
    hoverinfo="skip"
))
# Marcadores de los hitos
for h in hitos:
    n_h = int(np.ceil(len(b2b_sorted) * h / 100))
    pct_h = b2b_sorted.iloc[n_h - 1]["pct_acumulado"]
    fig.add_trace(go.Scatter(
        x=[h], y=[pct_h], mode="markers+text",
        marker=dict(color=COLOR_DESTACAR, size=11, line=dict(color="white", width=2)),
        text=[f"  {pct_h:.1f} %"], textposition="middle right",
        textfont=dict(color=COLOR_DESTACAR, size=11),
        showlegend=False, hoverinfo="skip"
    ))

fig.update_layout(
    title=dict(text="<b>Curva de Pareto · Concentración de la facturación B2B</b><br><sup>Porcentaje acumulado de facturación frente al porcentaje de clientes (excluyendo cuentas técnicas)</sup>"),
    xaxis_title="Porcentaje de clientes (ordenados de mayor a menor facturación)",
    yaxis_title="Porcentaje acumulado de facturación",
    height=550,
)
guardar_y_mostrar(fig, "06_curva_pareto_b2b")

CURVA DE PARETO B2B (sin cuentas técnicas)
   Top  10 % de clientes (  300) acumula el 57.91 % de la facturación
   Top  20 % de clientes (  600) acumula el 74.40 % de la facturación
   Top  50 % de clientes (1,499) acumula el 94.99 % de la facturación
   Top  80 % de clientes (2,399) acumula el 99.53 % de la facturación


   Figura guardada en: 06_curva_pareto_b2b.png


In [30]:
# 4.3 Top 20 clientes B2B (con cuentas técnicas, marcadas aparte)
top20_b2b = df.sort_values("facturacion_b2b", ascending=False).head(20).copy()
top20_b2b["es_tecnica_label"] = top20_b2b["es_cuenta_tecnica"].map({True: "Cuenta técnica", False: "Cliente regular"})
top20_b2b["display_name"] = top20_b2b["nombre_cliente"].str[:35]

fig = px.bar(
    top20_b2b.sort_values("facturacion_b2b"),
    x="facturacion_b2b", y="display_name",
    orientation="h",
    color="es_tecnica_label",
    color_discrete_map={"Cuenta técnica": COLOR_DESTACAR, "Cliente regular": COLOR_NACIONAL},
    hover_data={"display_name": False, "tipo_mercado": True, "pais": True,
                 "num_pedidos_b2b": ":,", "facturacion_b2b": ":,.2f"},
)
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Mercado: %{customdata[0]}<br>País: %{customdata[1]}<br>Pedidos: %{customdata[2]:,}<br>Facturación: %{x:,.2f} €<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Top 20 clientes B2B por facturación 2022-2025</b><br><sup>Cuentas técnicas (REGO, HERREROS, ECI) destacadas en dorado</sup>"),
    xaxis_title="Facturación B2B acumulada (€)",
    yaxis_title="",
    legend=dict(title=""),
    height=700,
)
guardar_y_mostrar(fig, "07_top20_clientes_b2b", h=700)

   Figura guardada en: 07_top20_clientes_b2b.png


In [31]:
# 4.4 Comparativa Nacional vs Internacional (B2B)
b2b_activos = df_sin_tecnicas[df_sin_tecnicas["facturacion_b2b"] > 0]
resumen_mercado = b2b_activos.groupby("tipo_mercado").agg(
    num_clientes=("id_cliente", "count"),
    facturacion_total=("facturacion_b2b", "sum"),
    facturacion_media=("facturacion_b2b", "mean"),
    facturacion_mediana=("facturacion_b2b", "median"),
    pedidos_totales=("num_pedidos_b2b", "sum"),
).reset_index()
resumen_mercado["ticket_medio_pedido"] = resumen_mercado["facturacion_total"] / resumen_mercado["pedidos_totales"]

print("COMPARATIVA NACIONAL VS INTERNACIONAL · CANAL B2B (sin cuentas técnicas)")
print("=" * 80)
for _, row in resumen_mercado.iterrows():
    print(f"\n   {row['tipo_mercado']}")
    print(f"      Clientes activos:    {fmt_es(row['num_clientes']):>10s}")
    print(f"      Facturación total:   {fmt_es(row['facturacion_total'], 2):>15s} €")
    print(f"      Facturación media:   {fmt_es(row['facturacion_media'], 2):>15s} €")
    print(f"      Facturación mediana: {fmt_es(row['facturacion_mediana'], 2):>15s} €")
    print(f"      Pedidos totales:     {fmt_es(row['pedidos_totales']):>15s}")
    print(f"      Ticket medio pedido: {fmt_es(row['ticket_medio_pedido'], 2):>15s} €")

# Visualización: cuatro subplots con métricas comparativas
from plotly.subplots import make_subplots
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=("Clientes activos", "Facturación total (M €)",
                    "Facturación mediana (€)", "Ticket medio pedido (€)"),
    horizontal_spacing=0.08,
)
colores_mer = [COLOR_NACIONAL if t == "NACIONAL" else COLOR_INTERNACIONAL for t in resumen_mercado["tipo_mercado"]]

fig.add_trace(go.Bar(x=resumen_mercado["tipo_mercado"], y=resumen_mercado["num_clientes"],
                     marker_color=colores_mer, text=resumen_mercado["num_clientes"].map(lambda v: f"{v:,}".replace(",", ".")),
                     textposition="outside", showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=resumen_mercado["tipo_mercado"], y=resumen_mercado["facturacion_total"]/1e6,
                     marker_color=colores_mer, text=(resumen_mercado["facturacion_total"]/1e6).round(1).astype(str) + " M",
                     textposition="outside", showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=resumen_mercado["tipo_mercado"], y=resumen_mercado["facturacion_mediana"],
                     marker_color=colores_mer, text=resumen_mercado["facturacion_mediana"].round(0).map(lambda v: f"{int(v):,} €".replace(",", ".")),
                     textposition="outside", showlegend=False), row=1, col=3)
fig.add_trace(go.Bar(x=resumen_mercado["tipo_mercado"], y=resumen_mercado["ticket_medio_pedido"],
                     marker_color=colores_mer, text=resumen_mercado["ticket_medio_pedido"].round(0).map(lambda v: f"{int(v):,} €".replace(",", ".")),
                     textposition="outside", showlegend=False), row=1, col=4)

fig.update_layout(
    title=dict(text="<b>Comparativa Nacional vs Internacional · Canal B2B</b><br><sup>Métricas clave por mercado (excluyendo cuentas técnicas)</sup>"),
    height=480, showlegend=False,
)
guardar_y_mostrar(fig, "08_comparativa_b2b_mercados")

COMPARATIVA NACIONAL VS INTERNACIONAL · CANAL B2B (sin cuentas técnicas)

   INTERNACIONAL
      Clientes activos:         1.141
      Facturación total:     28.536.091,93 €
      Facturación media:         25.009,72 €
      Facturación mediana:        4.415,20 €
      Pedidos totales:              49.065
      Ticket medio pedido:          581,60 €

   NACIONAL
      Clientes activos:         1.857
      Facturación total:     40.487.662,72 €
      Facturación media:         21.802,73 €
      Facturación mediana:        9.739,62 €
      Pedidos totales:             133.323
      Ticket medio pedido:          303,68 €


   Figura guardada en: 08_comparativa_b2b_mercados.png


### 4.5 Caracterización del Top 100 clientes B2B

Se examina la composición del Top 100 clientes B2B para entender su peso relativo sobre la facturación total y su distribución por mercado, perfil y tipo de cuenta.

In [32]:
# 4.5 Caracterización del Top 100 clientes B2B
top100_b2b = df.sort_values("facturacion_b2b", ascending=False).head(100).copy()

# Métricas globales
fb2b_top100 = top100_b2b["facturacion_b2b"].sum()
n_nacional = int((top100_b2b["tipo_mercado"] == "NACIONAL").sum())
n_internacional = int((top100_b2b["tipo_mercado"] == "INTERNACIONAL").sum())
n_cuentas_tec = int(top100_b2b["es_cuenta_tecnica"].sum())

print("CARACTERIZACIÓN DEL TOP 100 B2B")
print("=" * 70)
print(f"   Facturación acumulada del Top 100:  {fmt_es(fb2b_top100/1e6, 2):>8s} M €  ({100*fb2b_top100/df['facturacion_b2b'].sum():.2f} % del B2B total)")
print(f"   Clientes nacionales:                {n_nacional:>4d}  ({n_nacional}%)")
print(f"   Clientes internacionales:           {n_internacional:>4d}  ({n_internacional}%)")
print(f"   Cuentas técnicas en el Top 100:     {n_cuentas_tec:>4d}")

# Estadísticas del ticket promedio del Top 100
print(f"\n   Facturación B2B en el Top 100:")
print(f"      Mínimo:    {fmt_es(top100_b2b['facturacion_b2b'].min(), 2):>15s} €")
print(f"      Mediana:   {fmt_es(top100_b2b['facturacion_b2b'].median(), 2):>15s} €")
print(f"      Media:     {fmt_es(top100_b2b['facturacion_b2b'].mean(), 2):>15s} €")
print(f"      Máximo:    {fmt_es(top100_b2b['facturacion_b2b'].max(), 2):>15s} €")

# Visualización: distribución del Top 100 por mercado y tipo (sunburst)
top100_b2b["categoria"] = top100_b2b.apply(
    lambda r: "Cuenta técnica" if r["es_cuenta_tecnica"] else f"{r['tipo_mercado']} regular", axis=1
)
resumen = top100_b2b.groupby("categoria").agg(
    num_clientes=("id_cliente", "count"),
    facturacion=("facturacion_b2b", "sum"),
).reset_index().sort_values("facturacion", ascending=False)

mapa_color = {
    "Cuenta técnica":         COLOR_DESTACAR,
    "NACIONAL regular":       COLOR_NACIONAL,
    "INTERNACIONAL regular":  COLOR_INTERNACIONAL,
}
colores_resumen = [mapa_color.get(c, COLOR_NEUTRO) for c in resumen["categoria"]]

fig = go.Figure(data=[go.Pie(
    labels=resumen["categoria"],
    values=resumen["facturacion"],
    marker=dict(colors=colores_resumen, line=dict(color="white", width=2)),
    textinfo="label+percent+value",
    texttemplate="%{label}<br>%{percent}<br>%{value:,.0f} €",
    textposition="auto",
    textfont=dict(size=11, color="white"),
    hole=0.4,
)])
fig.update_layout(
    title=dict(text=f"<b>Composición de la facturación del Top 100 B2B</b><br><sup>Reparto entre cuentas técnicas, clientes nacionales y clientes internacionales · {fmt_es(fb2b_top100/1e6, 2)} M € totales</sup>"),
    height=520,
    annotations=[dict(text=f"<b>{fmt_es(fb2b_top100/1e6, 2)}</b><br>M €<br>Top 100", x=0.5, y=0.5,
                       font=dict(size=14), showarrow=False)],
)
guardar_y_mostrar(fig, "08bis_top100_b2b_composicion")

CARACTERIZACIÓN DEL TOP 100 B2B
   Facturación acumulada del Top 100:     42,87 M €  (50.32 % del B2B total)
   Clientes nacionales:                  59  (59%)
   Clientes internacionales:             41  (41%)
   Cuentas técnicas en el Top 100:        3

   Facturación B2B en el Top 100:
      Mínimo:          95.904,02 €
      Mediana:        137.885,35 €
      Media:          428.696,26 €
      Máximo:      10.812.150,42 €


   Figura guardada en: 08bis_top100_b2b_composicion.png


### Conclusiones del Top 100 B2B

El Top 100 de clientes B2B concentra el 50,32 % de la facturación total del canal mayorista en el cuatrienio 2022-2025, alcanzando los 42,87 millones de euros sobre el total de 85,2 millones. Esta cifra confirma una concentración moderadamente elevada del canal B2B, característica del modelo de distribución mayorista.

La composición del Top 100 muestra un equilibrio notable entre los mercados: 59 clientes nacionales y 41 clientes internacionales conviven en la franja de mayor valor, lo que sugiere que la estrategia comercial internacional ha logrado consolidar una base de clientes mayoristas comparable en peso a la nacional. Las tres cuentas técnicas identificadas (REGO, HERREROS y EL CORTE INGLES) aparecen también en este Top, lo que refuerza la conveniencia de tratarlas explícitamente en la fase de clustering del Capítulo 5.

La distribución estadística interna del Top 100 revela una asimetría positiva muy pronunciada: la mediana se sitúa en 137.885 € pero la media alcanza los 428.696 € debido al efecto del cliente máximo (EL CORTE INGLES, 10,8 M €). Este comportamiento es coherente con la curva de Pareto y justifica la aplicación de transformaciones logarítmicas o métodos robustos sobre las variables económicas en los análisis multivariantes posteriores.

## 5. Análisis económico Retail

In [33]:
# 5.1 Top 20 clientes Retail
top20_retail = df.sort_values("facturacion_retail_neta_eur", ascending=False).head(20).copy()
top20_retail["es_tecnica_label"] = top20_retail["es_cuenta_tecnica"].map({True: "Cuenta técnica", False: "Cliente regular"})
top20_retail["display_name"] = top20_retail["nombre_cliente"].str[:35]

fig = px.bar(
    top20_retail.sort_values("facturacion_retail_neta_eur"),
    x="facturacion_retail_neta_eur", y="display_name",
    orientation="h",
    color="es_tecnica_label",
    color_discrete_map={"Cuenta técnica": COLOR_DESTACAR, "Cliente regular": COLOR_NACIONAL},
    hover_data={"display_name": False, "tipo_mercado": True, "pais": True,
                 "num_operaciones": ":,", "facturacion_retail_neta_eur": ":,.2f"},
)
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Mercado: %{customdata[0]}<br>País: %{customdata[1]}<br>Operaciones: %{customdata[2]:,}<br>Facturación: %{x:,.2f} €<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Top 20 clientes Retail por facturación 2022-2025</b><br><sup>Cuentas técnicas destacadas en dorado</sup>"),
    xaxis_title="Facturación Retail neta acumulada (€)",
    yaxis_title="",
    legend=dict(title=""),
    height=700,
)
guardar_y_mostrar(fig, "09_top20_clientes_retail", h=700)

   Figura guardada en: 09_top20_clientes_retail.png


### 5.3 Distribución de la facturación Retail por país

Analizamos cómo se reparte la facturación del canal retail entre los distintos países de la cartera internacional para identificar los mercados de mayor peso fuera de España.

In [34]:
# 5.3 Distribución de la facturación Retail por país (sin cuentas técnicas)
retail_por_pais = (df_sin_tecnicas[df_sin_tecnicas["facturacion_retail_neta_eur"] > 0]
                   .groupby("pais")
                   .agg(num_clientes=("id_cliente", "count"),
                        facturacion=("facturacion_retail_neta_eur", "sum"),
                        ticket_medio=("ticket_medio_retail_eur", "mean"))
                   .reset_index()
                   .sort_values("facturacion", ascending=False))

top15_retail_pais = retail_por_pais.head(15).copy()
top15_retail_pais["pct_facturacion"] = (100 * top15_retail_pais["facturacion"]
                                         / retail_por_pais["facturacion"].sum()).round(2)

print("TOP 15 PAÍSES POR FACTURACIÓN RETAIL (sin cuentas técnicas)")
print("=" * 75)
for _, row in top15_retail_pais.iterrows():
    pais_str = str(row['pais']) if pd.notna(row['pais']) else "Sin identificar"
    print(f"   {pais_str:<25s}  {fmt_es(row['num_clientes']):>6s} cli  {fmt_es(row['facturacion']/1e6, 2):>7s} M €  ({row['pct_facturacion']:>5.2f} %)")

# Asignar color especial a España (color nacional) y resto en color internacional
top15_retail_pais["color"] = top15_retail_pais["pais"].apply(
    lambda p: COLOR_NACIONAL if str(p) == "España" else COLOR_INTERNACIONAL
)

fig = go.Figure(go.Bar(
    x=top15_retail_pais.sort_values("facturacion")["facturacion"] / 1e6,
    y=top15_retail_pais.sort_values("facturacion")["pais"].fillna("Sin identificar"),
    orientation="h",
    marker=dict(color=top15_retail_pais.sort_values("facturacion")["color"]),
    text=top15_retail_pais.sort_values("facturacion")["facturacion"].apply(lambda v: f"{v/1e6:.2f} M €"),
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Facturación: %{x:.2f} M €<extra></extra>",
))
fig.update_layout(
    title=dict(text="<b>Top 15 países por facturación Retail acumulada</b><br><sup>Cuatrienio 2022-2025 · sin cuentas técnicas · color rosa = España, azul = resto</sup>"),
    xaxis_title="Facturación Retail acumulada (M €)",
    yaxis_title="",
    height=600, showlegend=False,
)
guardar_y_mostrar(fig, "10bis_retail_por_pais", h=600)

TOP 15 PAÍSES POR FACTURACIÓN RETAIL (sin cuentas técnicas)
   España                      1.868 cli    35,64 M €  (53.11 %)
   Portugal                      296 cli     8,08 M €  (12.04 %)
   Italia                        516 cli     4,65 M €  ( 6.93 %)
   Israel                          1 cli     2,51 M €  ( 3.73 %)
   México                          3 cli     2,41 M €  ( 3.59 %)
   Reino Unido                     5 cli     2,32 M €  ( 3.46 %)
   Rusia                           3 cli     2,11 M €  ( 3.14 %)
   Bélgica                        47 cli     1,55 M €  ( 2.32 %)
   Polonia                        70 cli     1,05 M €  ( 1.56 %)
   Austria                         2 cli     0,83 M €  ( 1.23 %)
   Kuwait                          1 cli     0,64 M €  ( 0.95 %)
   República Checa                 1 cli     0,54 M €  ( 0.81 %)
   Chile                           1 cli     0,51 M €  ( 0.76 %)
   Alemania                        2 cli     0,46 M €  ( 0.69 %)
   Grecia                     

   Figura guardada en: 10bis_retail_por_pais.png


### Conclusiones del análisis Retail por país

La distribución geográfica de la facturación del canal retail revela hallazgos especialmente relevantes para la estrategia comercial internacional de Selmark. España concentra el 53,11 % del retail con 1.868 clientes activos, lo que constituye la base natural del negocio doméstico. Portugal se confirma como el segundo mercado por volumen con 8,08 millones de euros y 296 clientes, un volumen significativo coherente con la proximidad geográfica y cultural.

El hallazgo más relevante de este análisis es la existencia de un grupo de países con una intensidad de facturación por cliente extraordinariamente elevada, característica de mercados servidos a través de distribuidores únicos o casi únicos. Israel genera 2,51 millones de euros con un solo cliente, México alcanza los 2,41 millones con apenas tres clientes, Reino Unido genera 2,32 millones con cinco, y Rusia 2,11 millones con tres clientes. Estos cuatro países, con un total de doce clientes únicos, aportan conjuntamente cerca de 9,4 millones de euros, una cifra superior a la generada por Polonia con setenta clientes activos.

Esta dualidad estructural —mercados de distribución masiva frente a mercados servidos por distribuidores únicos— constituye un hallazgo clave para el Capítulo 6 (geomarketing) y justifica que las estrategias comerciales propuestas en el Capítulo 8 contemplen diferenciaciones explícitas por tipología de mercado. La identificación de estos distribuidores únicos también será relevante para el análisis de churn del Capítulo 7, dado que su eventual pérdida implicaría un impacto desproporcionado sobre la facturación internacional.

In [35]:
# 5.2 Devoluciones y descuentos - análisis agregado
print("ANÁLISIS DE DEVOLUCIONES Y DESCUENTOS (canal retail, todos los clientes)")
print("=" * 75)

total_bruto = df["facturacion_retail_bruta_eur"].sum()
total_neto = df["facturacion_retail_neta_eur"].sum()
total_devol = df["importe_devoluciones_eur"].sum()
total_desc = df["descuento_total_eur"].sum()

print(f"   Facturación bruta total:      {fmt_es(total_bruto, 2):>20s} €")
print(f"   Facturación neta total:       {fmt_es(total_neto, 2):>20s} €")
print(f"   Importe total devoluciones:   {fmt_es(total_devol, 2):>20s} € ({100*total_devol/total_neto:.2f} % sobre neta)")
print(f"   Importe total descuentos:     {fmt_es(total_desc, 2):>20s} € ({100*total_desc/total_neto:.2f} % sobre neta)")

# Visualización: % devoluciones por cliente activo (boxplot por mercado)
retail_activos = df[(df["facturacion_retail_neta_eur"] > 1000) & (~df["es_cuenta_tecnica"])].copy()
retail_activos["pct_devol_sobre_neta"] = (100 * retail_activos["importe_devoluciones_eur"]
                                            / retail_activos["facturacion_retail_neta_eur"].replace(0, np.nan))
retail_activos = retail_activos[retail_activos["pct_devol_sobre_neta"].between(0, 50)]

fig = px.box(
    retail_activos, x="tipo_mercado", y="pct_devol_sobre_neta",
    color="tipo_mercado",
    color_discrete_map={"NACIONAL": COLOR_NACIONAL, "INTERNACIONAL": COLOR_INTERNACIONAL},
    points="outliers",
)
fig.update_layout(
    title=dict(text="<b>Tasa de devoluciones por cliente · Canal Retail</b><br><sup>Distribución de la tasa (devoluciones / facturación neta), clientes con > 1.000 € facturados, sin cuentas técnicas</sup>"),
    xaxis_title="",
    yaxis_title="Tasa de devolución (% sobre facturación neta)",
    height=500, showlegend=False,
)
guardar_y_mostrar(fig, "10_devoluciones_por_mercado")

ANÁLISIS DE DEVOLUCIONES Y DESCUENTOS (canal retail, todos los clientes)
   Facturación bruta total:             85.187.086,81 €
   Facturación neta total:              84.945.543,36 €
   Importe total devoluciones:           4.450.414,71 € (5.24 % sobre neta)
   Importe total descuentos:               241.543,45 € (0.28 % sobre neta)


   Figura guardada en: 10_devoluciones_por_mercado.png


## 6. Cruce B2B × Retail

Un cliente puede operar simultáneamente en ambos canales. Esta sección caracteriza cuántos clientes pertenecen a cada combinación posible y el volumen económico de cada segmento.

In [36]:
# 6.1 Distribución de clientes por combinación de canales
df["segmento_canal"] = np.select(
    [
        df["tiene_facturacion_b2b"] & df["tiene_actividad_minorista"],
        df["tiene_facturacion_b2b"] & ~df["tiene_actividad_minorista"],
        ~df["tiene_facturacion_b2b"] & df["tiene_actividad_minorista"],
    ],
    ["B2B + Retail", "Solo B2B", "Solo Retail"],
    default="Sin actividad económica",
)

dist_canal = df.groupby("segmento_canal").agg(
    num_clientes=("id_cliente", "count"),
    fact_b2b_total=("facturacion_b2b", "sum"),
    fact_retail_total=("facturacion_retail_neta_eur", "sum"),
).reset_index()
dist_canal["facturacion_combinada"] = dist_canal["fact_b2b_total"] + dist_canal["fact_retail_total"]
dist_canal["pct_clientes"] = (100 * dist_canal["num_clientes"] / dist_canal["num_clientes"].sum()).round(2)
dist_canal = dist_canal.sort_values("num_clientes", ascending=False).reset_index(drop=True)

print("DISTRIBUCIÓN DE CLIENTES POR COMBINACIÓN DE CANALES")
print("=" * 75)
for _, row in dist_canal.iterrows():
    print(f"   {row['segmento_canal']:<28s}  {fmt_es(row['num_clientes']):>6s} cli ({row['pct_clientes']:>5.2f} %)  Facturación total: {fmt_es(row['facturacion_combinada']/1e6, 2):>6s} M €")

# ─── SUNBURST CORREGIDO ──────────────────────────────────────────────
# Asignar color a cada segmento
mapa_colores = {
    "B2B + Retail":               COLOR_DESTACAR,
    "Solo B2B":                   COLOR_NACIONAL,
    "Solo Retail":                COLOR_INTERNACIONAL,
    "Sin actividad económica":    COLOR_NEUTRO,
}
colores_segmentos = [mapa_colores.get(s, COLOR_NEUTRO) for s in dist_canal["segmento_canal"]]

# Construcción correcta: padre vacío sin nodo raíz explícito
fig = go.Figure(go.Sunburst(
    labels=dist_canal["segmento_canal"].tolist(),
    parents=["" for _ in range(len(dist_canal))],
    values=dist_canal["num_clientes"].tolist(),
    marker=dict(colors=colores_segmentos, line=dict(color="white", width=2)),
    textinfo="label+percent entry+value",
    insidetextorientation="radial",
    textfont=dict(size=13, color="white"),
    hovertemplate="<b>%{label}</b><br>Clientes: %{value:,}<br>Porcentaje: %{percentEntry}<extra></extra>",
))
fig.update_layout(
    title=dict(text="<b>Distribución de la cartera por combinación de canales</b><br><sup>3.492 clientes clasificados según su actividad en B2B y/o Retail</sup>"),
    height=600,
    margin=dict(t=90, l=40, r=40, b=40),
)
guardar_y_mostrar(fig, "11_sunburst_canales", w=700, h=600)

DISTRIBUCIÓN DE CLIENTES POR COMBINACIÓN DE CANALES
   B2B + Retail                   3.050 cli (87.34 %)  Facturación total: 170,14 M €
   Sin actividad económica          405 cli (11.60 %)  Facturación total:   0,00 M €
   Solo B2B                          33 cli ( 0.95 %)  Facturación total:   0,01 M €
   Solo Retail                        4 cli ( 0.11 %)  Facturación total:   0,00 M €


   Figura guardada en: 11_sunburst_canales.png


## 7. Síntesis económica

In [37]:
# Resumen ejecutivo del análisis económico
print("=" * 75)
print("SÍNTESIS ECONÓMICA DEL ANÁLISIS DESCRIPTIVO 09a")
print("=" * 75)

print(f"\n1. COMPOSICIÓN DE LA CARTERA")
print(f"   Total clientes:                 {fmt_es(len(df)):>10s}")
print(f"   Cartera nacional:               {fmt_es(int((df['tipo_mercado']=='NACIONAL').sum())):>10s}")
print(f"   Cartera internacional:          {fmt_es(int((df['tipo_mercado']=='INTERNACIONAL').sum())):>10s}")
print(f"   Cuentas técnicas identificadas: {len(ids_cuentas_tecnicas):>10d}")

print(f"\n2. ACTIVIDAD ECONÓMICA TOTAL DEL CUATRIENIO 2022-2025")
fb2b = df['facturacion_b2b'].sum()
fret = df['facturacion_retail_neta_eur'].sum()
ftot = fb2b + fret
print(f"   Facturación B2B:                {fmt_es(fb2b/1e6, 2):>8s} M €")
print(f"   Facturación Retail neta:        {fmt_es(fret/1e6, 2):>8s} M €")
print(f"   Facturación TOTAL documentada:  {fmt_es(ftot/1e6, 2):>8s} M €")

print(f"\n3. CONCENTRACIÓN — Vista con todas las cuentas (incluye técnicas)")
top10_b2b = df.sort_values("facturacion_b2b", ascending=False).head(10)["facturacion_b2b"].sum()
top10_ret = df.sort_values("facturacion_retail_neta_eur", ascending=False).head(10)["facturacion_retail_neta_eur"].sum()
print(f"   Top 10 clientes B2B aportan:        {100*top10_b2b/fb2b:>6.2f} % de la facturación B2B")
print(f"   Top 10 clientes Retail aportan:     {100*top10_ret/fret:>6.2f} % de la facturación Retail")

print(f"\n   CONCENTRACIÓN — Vista sin cuentas técnicas (lectura honesta del mercado real)")
fb2b_st = df_sin_tecnicas['facturacion_b2b'].sum()
fret_st = df_sin_tecnicas['facturacion_retail_neta_eur'].sum()
top10_b2b_st = df_sin_tecnicas.sort_values("facturacion_b2b", ascending=False).head(10)["facturacion_b2b"].sum()
top10_ret_st = df_sin_tecnicas.sort_values("facturacion_retail_neta_eur", ascending=False).head(10)["facturacion_retail_neta_eur"].sum()
print(f"   Top 10 clientes B2B aportan:        {100*top10_b2b_st/fb2b_st:>6.2f} % de la facturación B2B")
print(f"   Top 10 clientes Retail aportan:     {100*top10_ret_st/fret_st:>6.2f} % de la facturación Retail")

print(f"\n   Impacto agregado de las 3 cuentas técnicas:")
fb2b_tec = df[df['es_cuenta_tecnica']]['facturacion_b2b'].sum()
fret_tec = df[df['es_cuenta_tecnica']]['facturacion_retail_neta_eur'].sum()
print(f"   - Facturación B2B de técnicas:      {fmt_es(fb2b_tec/1e6, 2):>6s} M €  ({100*fb2b_tec/fb2b:>5.2f} % del B2B total)")
print(f"   - Facturación retail de técnicas:   {fmt_es(fret_tec/1e6, 2):>6s} M €  ({100*fret_tec/fret:>5.2f} % del Retail total)")

print(f"\n4. SEGMENTACIÓN POR ACTIVIDAD DE CANAL")
for _, row in dist_canal.iterrows():
    print(f"   {row['segmento_canal']:<28s}: {fmt_es(row['num_clientes']):>6s} clientes ({row['pct_clientes']:>5.2f} %)")

print(f"\n" + "=" * 75)
print(f"CIERRE DEL NOTEBOOK 09a · Continúa en 09b (temporal y sociodemográfico)")
print(f"Figuras generadas: {len(list(RUTA_FIGURAS.glob('*.png')))} archivos PNG en {RUTA_FIGURAS.name}/")
print("=" * 75)

SÍNTESIS ECONÓMICA DEL ANÁLISIS DESCRIPTIVO 09a

1. COMPOSICIÓN DE LA CARTERA
   Total clientes:                      3.492
   Cartera nacional:                    2.105
   Cartera internacional:               1.387
   Cuentas técnicas identificadas:          3

2. ACTIVIDAD ECONÓMICA TOTAL DEL CUATRIENIO 2022-2025
   Facturación B2B:                   85,20 M €
   Facturación Retail neta:           84,95 M €
   Facturación TOTAL documentada:    170,14 M €

3. CONCENTRACIÓN — Vista con todas las cuentas (incluye técnicas)
   Top 10 clientes B2B aportan:         32.00 % de la facturación B2B
   Top 10 clientes Retail aportan:      35.28 % de la facturación Retail

   CONCENTRACIÓN — Vista sin cuentas técnicas (lectura honesta del mercado real)
   Top 10 clientes B2B aportan:         18.07 % de la facturación B2B
   Top 10 clientes Retail aportan:      20.52 % de la facturación Retail

   Impacto agregado de las 3 cuentas técnicas:
   - Facturación B2B de técnicas:       16,20 M €  (19.0

### Conclusiones generales del análisis económico

El análisis descriptivo económico desarrollado en este notebook documenta una facturación total de 170,14 millones de euros durante el cuatrienio 2022-2025, distribuida de forma sorprendentemente equilibrada entre los dos canales operativos de Selmark: 85,20 millones del canal B2B mayorista y 84,95 millones del canal retail minorista. Este equilibrio de magnitudes, unido a la cifra del 87,34 % de clientes que operan simultáneamente en ambos canales, configura un modelo de negocio profundamente dual en el que la mayoría de la cartera funciona como nexo entre la distribución profesional y la venta al consumidor final.

La concentración del negocio se ha presentado en doble lectura para ofrecer una caracterización honesta del mercado. La vista que incluye todas las cuentas muestra un Top 10 que concentra el 32 % del B2B y el 35 % del retail, valores que aparentan una concentración elevada. Sin embargo, la exclusión de las tres cuentas técnicas identificadas (REGO, HERREROS y EL CORTE INGLES) revela que la concentración real del mercado se sitúa en el 18,07 % del B2B y el 20,52 % del retail. Esta diferencia, de aproximadamente catorce puntos porcentuales, evidencia el impacto desproporcionado de las cuentas técnicas sobre los agregados y justifica plenamente su tratamiento diferenciado en los análisis posteriores.

La caracterización geográfica del canal retail aporta un hallazgo de naturaleza estratégica: la coexistencia en la cartera internacional de mercados de distribución masiva (Italia, Polonia, Bélgica) con mercados de distribución concentrada en un número muy reducido de clientes (Israel, México, Reino Unido, Rusia). Esta dualidad estructural será determinante para el diseño de la segmentación y para la formulación de recomendaciones comerciales diferenciadas.

### Hallazgos para los siguientes capítulos

Los hallazgos del análisis descriptivo económico orientan las decisiones metodológicas y comerciales de los capítulos sucesivos del proyecto:

- **Capítulo 5 (Clustering).** Las tres cuentas técnicas identificadas (REGO, HERREROS y EL CORTE INGLES) deberán excluirse explícitamente del algoritmo de clustering para evitar distorsiones. La asimetría pronunciada de la facturación, evidenciada por la divergencia entre mediana (137.885 €) y media (428.696 €) en el Top 100, justifica la aplicación de transformaciones logarítmicas o el uso de escalado robusto sobre las variables económicas. El equilibrio de la cartera entre clientes nacionales e internacionales en el Top 100 sugiere que el clustering pueda producir segmentos transversales a la geografía o, alternativamente, que el tipo de mercado deba incorporarse como variable explícita del modelo.

- **Capítulo 6 (Geomarketing).** La dualidad estructural detectada entre mercados de distribución masiva y mercados de distribuidor único obliga a abordar el análisis territorial internacional con escalas y representaciones diferenciadas. Para el mercado nacional, la concentración detectada por provincias servirá como base para el cruce con los perfiles MOSAIC en el siguiente notebook.

- **Capítulo 7 (Churn).** Los clientes únicos en mercados extranjeros (Israel, México, Reino Unido, Rusia) constituyen casos de impacto desproporcionado: su eventual pérdida supondría una caída significativa de la facturación internacional. El análisis de churn deberá identificarlos como casos de máxima prioridad estratégica.

- **Capítulo 8 (Recomendaciones y ROI).** El hallazgo de que el 11,60 % de la cartera no presenta actividad económica documentada en el periodo de análisis (405 clientes inactivos) abre una línea natural de recomendación comercial orientada a su reactivación, cuyo retorno potencial se cuantificará en el análisis de ROI del último capítulo.

In [38]:
con.close()
print("Conexión a DuckDB cerrada. Notebook 09a completado.")

Conexión a DuckDB cerrada. Notebook 09a completado.
